In [ ]:
from FotR import GANDALF
import os

data_dir = '/home/airbus/CETACEO_cp_interp/DATA/'

import pandas as pd, numpy as np
mach = np.linspace(0.3, 0.8, 6) #[0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
Re = np.linspace(9.28e5, 1.97e6, 6)
Tt = 300
Pt = 1e5
D = 0.14

vinf = mach*np.sqrt(1.4*287*Tt)
rho = Pt/(Tt*287)
mu_check = 1.825e-5 # GANDALF.Backpack.Sutherland_law(mu0 = 1.716e-5, T=Tt, Treference=255.55)

Re_check = vinf*D*rho/mu_check

mu = (rho*mach*np.sqrt(1.4*287*Tt)*D)/Re

df_cases_bound = pd.DataFrame(data = mach, columns =['M'], dtype=np.float32)


In [ ]:
gdf = GANDALF(
    os.path.join(data_dir, "Cylinder_PINNS"),
    eq_type="rans", 
    num_stages = 2,
    version="flowsimulator2024"
)


gdf.define_cases(
    method="external",
    bounds=None,
    n_samples=None,
    peak_ranges=None,
    seed=None,
    external_dataframe = df_cases_bound
)

gdf.compute_param(
    name = 'Re',
    formula = 'Re',
    externals = {'Re': Re}
)

gdf.compute_param(
    name = 'mu',
    formula = 'mu',
    externals = {'mu': mu}
)

print(gdf.df_cases.columns)

In [ ]:
gdf.generate_folders(
    base_files = ["run_sst_v4.py", "run.sh"],
    mesh_path=os.path.join(data_dir, "Cylinder_PINNS", "sources", "mesh_cylinder_v1.msh"),
    script_dir=os.path.join(data_dir, "Cylinder_PINNS", "sources"),
    folder_fmt = "M_{M:.1f}",#"aoa_{AoA:.4f}_mach_{Mach:.4f}",
    overwrite=False,
    update_base_files=True,
    data_to_update={  # Esto debería cogerse de df_cases
        "MU_PLACEHOLDER": "mu",
        "LENGTH_PLACEHOLDER": str(D),
        "MACH_PLACEHOLDER": "M",
        "RE_PLACEHOLDER": "Re",
        "PYTHON_FILE_PLACEHOLDER": "run_sst_v4.py",
    }
)

In [ ]:
gdf.df_cases.to_csv(os.path.join(gdf.root_dir,'metadata', 'df_cases.csv'), index=False)

gdf.assign_jobs(
    file_sh="run.sh",
    nodes=[f'n00{n}' for n in [5, 6]],
    cpus_per_job=48,
    submit=True,
    )

In [ ]:
gdf.submit_cases()